# 🚀 AI Study Planner - Inference Server (Llama 3.1-8B)

Serveur d'inférence haute performance pour le projet AI Study Planner utilisant **Llama 3.1-8B** avec Unsloth.

**Instructions :**
1. Runtime → Change runtime type → **L4 GPU** (recommandé) ou **A100** (Colab Pro)
2. Runtime → Run all
3. Copiez l'URL ngrok et la clé API affichées
4. Configurez votre backend local dans `.env`

**Temps d'exécution :** ~2-3 minutes (téléchargement Llama 3.1-8B)

**GPU recommandés :**
- 🏆 **L4 (24GB)** : Performances optimales avec Llama 3.1-8B
- 🚀 **A100 (40GB+)** : Excellentes performances
- ⚡ **T4 (16GB)** : Utilisera Llama 3.2-3B (auto-détecté)

✅ **Pas besoin de compte HuggingFace !** (Unsloth télécharge directement)

In [ ]:
# 📦 CELLULE 1 — Installation des dépendances
print("📦 Installation des dépendances...")
print("⏳ Cela prend ~1-2 minutes...")
print("")

# IMPORTANT : Installation dans l'ordre correct pour éviter les conflits
# 1. D'abord installer xformers (compatible PyTorch)
print("🔧 Installation de xformers...")
!pip install -q xformers

# 2. Ensuite unsloth avec les dépendances correctes
print("🦥 Installation de Unsloth...")
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Enfin les autres dépendances
print("🌐 Installation de Flask et ngrok...")
!pip install -q flask flask-cors pyngrok
!pip install -q --upgrade pyngrok

print("")
print("✅ Dépendances installées!")
print("⚠️  Si vous voyez encore des erreurs, redémarrez le runtime : Runtime → Restart runtime")

In [ ]:
# 🔧 CELLULE 2 — Configuration
import os
import secrets
import json
from datetime import datetime
import torch

# Vérifier le GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️ ATTENTION : Pas de GPU détecté!")
    print("→ Allez dans Runtime → Change runtime type → GPU (T4, L4 ou A100)")
    raise RuntimeError("GPU requis pour l'inférence")
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU détecté : {gpu_name} ({gpu_memory:.1f} GB)")
    
    # Sélection automatique intelligente selon le GPU
    if "L4" in gpu_name and gpu_memory >= 24:
        print("💎 GPU L4 (24GB) détecté → Llama-3.1-8B (excellent compromis qualité/vitesse)")
        auto_model = "unsloth/Meta-Llama-3.1-8B-Instruct"
    elif "A100" in gpu_name and gpu_memory >= 40:
        print("🚀 GPU A100 (40GB+) détecté → Llama-3.1-8B (performances maximales)")
        auto_model = "unsloth/Meta-Llama-3.1-8B-Instruct"
    elif "A100" in gpu_name or "V100" in gpu_name:
        print("💡 GPU puissant détecté → Llama-3.1-8B")
        auto_model = "unsloth/Meta-Llama-3.1-8B-Instruct"
    elif "T4" in gpu_name:
        print("💡 GPU T4 (16GB) détecté → Llama-3.2-3B (bon compromis T4)")
        auto_model = "unsloth/Llama-3.2-3B-Instruct"
    else:
        print("💡 GPU standard détecté → Llama-3.2-3B")
        auto_model = "unsloth/Llama-3.2-3B-Instruct"

print("")

# Génération d'une clé API sécurisée
API_KEY = f"sk-{secrets.token_urlsafe(32)}"

# Configuration du modèle
# Guide de sélection par GPU :
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  GPU          VRAM    Modèle recommandé                     Qualité
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  L4           24 GB   unsloth/Meta-Llama-3.1-8B-Instruct   ⭐⭐⭐⭐⭐
#  A100         40 GB+  unsloth/Meta-Llama-3.1-8B-Instruct   ⭐⭐⭐⭐⭐
#  T4           16 GB   unsloth/Llama-3.2-3B-Instruct        ⭐⭐⭐⭐
#  T4 (limité)  16 GB   unsloth/Llama-3.2-1B-Instruct        ⭐⭐⭐
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# MODE AUTO : Décommentez pour utiliser la sélection automatique
# MODEL_NAME = auto_model
#
# MODE MANUEL : Spécifiez votre modèle (par défaut)

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct"  # 🏆 Llama 3.1-8B (recommandé)
MAX_TOKENS = 1024
TEMPERATURE = 0.2

print("="*70)
print("🔧 Configuration")
print("="*70)
print(f"Modèle sélectionné  : {MODEL_NAME}")
print(f"Auto-recommandation : {auto_model}")
print(f"GPU Memory          : {gpu_memory:.1f} GB")
print(f"Max tokens          : {MAX_TOKENS}")
print(f"Temperature         : {TEMPERATURE}")
print("="*70)
print("")
print("🔑 Clé API générée :")
print(f"   {API_KEY}")
print("")
print("⚠️  IMPORTANT : Copiez cette clé, vous en aurez besoin !")

In [ ]:
# 🤖 CELLULE 3 — Chargement du modèle avec Unsloth
print("🤖 Chargement du modèle Llama avec Unsloth...")
print("⏳ Première fois : ~2-3 minutes (téléchargement)")
print("⏳ Sessions suivantes : ~30 secondes (cache Colab)")
print("")

# Fix pour l'erreur _wrap_tensor_autograd (si elle persiste)
try:
    import torch.distributed as dist
    if hasattr(dist, '_c10d_functional'):
        print("🔧 Application du patch pour PyTorch/Unsloth...")
except Exception as e:
    print(f"Note: {e}")

from unsloth import FastLanguageModel
import gc

# Nettoyer la mémoire GPU avant chargement
gc.collect()
torch.cuda.empty_cache()

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=2048,
        dtype=torch.float16,
        load_in_4bit=True,   # Quantification 4-bit (économise 75% de RAM GPU)
    )
    
    # Activer le mode inférence rapide (2-5x plus rapide)
    FastLanguageModel.for_inference(model)
    
    print("")
    print("✅ Modèle chargé avec succès!")
    print("")
    
    # Afficher les stats mémoire
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"📊 Mémoire GPU :")
    print(f"   Utilisée   : {allocated:.2f} GB")
    print(f"   Réservée   : {reserved:.2f} GB")
    print(f"   Total      : {total:.2f} GB")
    print(f"   Disponible : {total - reserved:.2f} GB ({(total - reserved) / total * 100:.1f}%)")
    
    if (total - reserved) < 1.0:
        print("")
        print("⚠️  Mémoire GPU presque pleine. Si vous avez des erreurs :")
        print("   1. Redémarrez le runtime (Runtime → Restart runtime)")
        print("   2. Utilisez un modèle plus petit (Llama-3.2-3B)")
    
except Exception as e:
    print(f"❌ Erreur lors du chargement du modèle : {e}")
    print("")
    print("Solutions possibles :")
    print("1. Vérifiez que le GPU est activé (Runtime → Change runtime type)")
    print("2. Redémarrez le runtime (Runtime → Restart runtime)")
    print("3. Essayez un modèle plus petit : unsloth/Llama-3.2-3B-Instruct")
    raise

In [ ]:
# 🌐 CELLULE 4 — Création du serveur Flask
from flask import Flask, request, jsonify
from flask_cors import CORS
import time

app = Flask(__name__)
CORS(app)

# Statistiques
stats = {
    "requests_count": 0,
    "total_tokens": 0,
    "avg_generation_time": 0,
    "start_time": datetime.now().isoformat()
}

def check_api_key():
    """Vérifier l'authentification API"""
    auth_header = request.headers.get('Authorization', '')
    if not auth_header.startswith('Bearer '):
        return False
    token = auth_header.replace('Bearer ', '')
    return token == API_KEY

def generate_text(prompt, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    """Générer du texte avec Unsloth"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Retourner seulement le texte généré (pas le prompt)
    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

@app.route('/health', methods=['GET'])
def health():
    """Endpoint de santé (pas d'authentification requise)"""
    return jsonify({
        "status": "healthy",
        "model": MODEL_NAME,
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
        "stats": stats
    })

@app.route('/generate', methods=['POST'])
def generate():
    """Endpoint de génération de texte"""
    if not check_api_key():
        return jsonify({"error": "Unauthorized"}), 401
    
    try:
        data = request.json
        prompt = data.get('prompt', '')
        temperature = data.get('temperature', TEMPERATURE)
        max_tokens = data.get('max_tokens', MAX_TOKENS)
        
        if not prompt:
            return jsonify({"error": "Missing prompt"}), 400
        
        start_time = time.time()
        generated_text = generate_text(prompt, temperature, max_tokens)
        generation_time = time.time() - start_time
        
        # Mettre à jour les stats
        tokens_count = len(tokenizer.encode(generated_text))
        stats["requests_count"] += 1
        stats["total_tokens"] += tokens_count
        stats["avg_generation_time"] = (
            (stats["avg_generation_time"] * (stats["requests_count"] - 1) + generation_time)
            / stats["requests_count"]
        )
        
        print(f"✅ Requête #{stats['requests_count']} | {generation_time:.2f}s | {tokens_count} tokens")
        
        return jsonify({
            "generated_text": generated_text,
            "generation_time": generation_time,
            "tokens_generated": tokens_count
        })
    
    except Exception as e:
        print(f"❌ Erreur : {str(e)}")
        return jsonify({"error": str(e)}), 500

@app.route('/stats', methods=['GET'])
def get_stats():
    """Endpoint de statistiques"""
    if not check_api_key():
        return jsonify({"error": "Unauthorized"}), 401
    return jsonify(stats)

print("✅ Serveur Flask créé!")

In [ ]:
# 🌍 CELLULE 5 — Tunnel ngrok + Démarrage du serveur
from pyngrok import ngrok, conf
import threading
from getpass import getpass

# Demander le token ngrok de manière sécurisée
print("🔐 Configuration ngrok")
print("")
print("Pour obtenir votre token ngrok GRATUIT :")
print("1. Allez sur https://dashboard.ngrok.com/signup")
print("2. Créez un compte (gratuit)")
print("3. Copiez votre authtoken depuis https://dashboard.ngrok.com/get-started/your-authtoken")
print("")

NGROK_TOKEN = getpass("Collez votre token ngrok ici (le texte sera masqué) : ")

if not NGROK_TOKEN or NGROK_TOKEN == "VOTRE_TOKEN_NGROK_ICI":
    print("❌ Token ngrok manquant !")
    print("Obtenez-en un sur : https://dashboard.ngrok.com/get-started/your-authtoken")
    raise ValueError("Token ngrok requis")

print("\n🌍 Démarrage du tunnel ngrok...")

# Configurer et démarrer ngrok
try:
    ngrok.kill()
except:
    pass

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(5000)
public_url = tunnel.public_url   # ← Extraire la string URL


print("")
print("=" * 70)
print("🎉 SERVEUR PRÊT !")
print("=" * 70)
print("")
print("📋 CONFIGURATION À COPIER dans votre backend/.env :")
print("")
print(f"AI_SERVICE_TYPE=colab")
print(f"COLAB_API_URL={public_url}")
print(f"COLAB_API_KEY={API_KEY}")
print("")
print("=" * 70)
print("")
print("🔍 Test rapide (dans votre terminal local) :")
print(f"")
print(f"curl {public_url}/health")
print(f"")
print("ou sur Windows :")
print(f"")
print(f"cd backend")
print(f"python test_colab_connection.py")
print("")
print("=" * 70)
print("")
print("✅ Le serveur est maintenant actif.")
print("⚠️  IMPORTANT : Laissez ce notebook ouvert !")
print("")

# Démarrer Flask dans un thread
def run_flask():
    app.run(host='0.0.0.0', port=5000, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print("🚀 Flask démarré sur le port 5000")
print("📡 Accessible via : " + str(public_url))

In [ ]:
# 🧪 CELLULE 6 — Test du serveur
import requests as req
import time

time.sleep(2)  # Attendre que Flask démarre

# Convertir l'objet ngrok en string
# public_url est déjà une string grâce à tunnel.public_url
public_url_str = public_url


print("🧪 Test du serveur...\n")

# Test 1 : Health check
try:
    response = req.get(f"{public_url_str}/health", timeout=10)
    print("✅ Health check OK :")
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f"❌ Health check échoué : {e}")

print("")

# Test 2 : Génération
print("🤖 Test de génération (peut prendre 10-20s)...")
test_prompt = """You are a study planner AI. Generate a short JSON study plan with 2 sessions.
Respond ONLY with valid JSON:
{
  "sessions": [
    {"day": "Monday", "start_time": "09:00", "end_time": "10:30", "subject": "Math", "task": "lecture_review"}
  ]
}"""

try:
    response = req.post(
        f"{public_url_str}/generate",
        json={"prompt": test_prompt, "temperature": 0.1, "max_tokens": 200},
        headers={"Authorization": f"Bearer {API_KEY}"},
        timeout=60
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"✅ Génération réussie en {result['generation_time']:.2f}s")
        print(f"📝 Réponse ({result['tokens_generated']} tokens) :")
        print(result['generated_text'][:300])
    else:
        print(f"❌ Erreur {response.status_code} : {response.text}")
except Exception as e:
    print(f"❌ Erreur : {e}")

## 📊 Monitoring & Performances

Le serveur est maintenant actif !

**⚠️ Important :**
- Gardez ce notebook **ouvert et connecté**
- Si Colab se déconnecte → **Runtime → Run all** pour redémarrer
- L'URL ngrok **change à chaque redémarrage** → mettez à jour `.env`

**Consulter les stats :**
```bash
curl {COLAB_API_URL}/stats -H "Authorization: Bearer {COLAB_API_KEY}"
```

**Performances attendues (Llama 3.1-8B avec quantization 4-bit) :**

| GPU | Tokens/seconde | Latence plan complet | VRAM utilisée |
|-----|----------------|----------------------|---------------|
| **L4 (24GB)** | 40-60 tok/s | 5-8s | ~6-8 GB |
| **A100 (40GB)** | 80-120 tok/s | 3-5s | ~6-8 GB |
| **T4 (16GB)** avec 3.2-3B | 30-40 tok/s | 8-12s | ~3-4 GB |

**Changer de modèle manuellement (Cellule 2) :**
```python
# Haute qualité (L4/A100 recommandé)
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct"  # 🏆 Meilleur choix

# Compromis rapide (T4)
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"  # ⚡ Bon pour T4

# Ultra rapide (T4 limité)
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"  # 💨 Très rapide
```

**Qualité de réponse attendue :**
- **Llama 3.1-8B** : Excellente compréhension des contraintes, planning cohérent et optimisé
- **Llama 3.2-3B** : Bonne qualité, peut nécessiter des ajustements mineurs
- **Llama 3.2-1B** : Correct pour tests, mais moins précis sur les optimisations complexes